# Lab 5 — Gradient boosting (CatBoost) for time series

**Goal:** treat forecasting as **supervised regression** on **lag and calendar features**, train **CatBoost** on the past only, and evaluate on a **time-ordered** test split. Compare with a linear baseline if useful.

**Install:** `pip install catboost` (CPU build is enough for this lab).

**Prerequisite:** `generate_data.ipynb`.

Context from `time_series_forecasting.md`: many engineered features; tree models capture **interactions** and **nonlinear** calendar effects; still respect **causal** features and honest validation.

## 1. Why boosting for series?

Gradient boosting builds an additive model of **weak learners** (trees). With **lag inputs**, trees can approximate nonlinear dynamics without hand-writing ARMA orders. Risks: **overfitting** on small samples, **leakage** if future information slips into features, and poor **long-horizon** behaviour unless you model each horizon separately or use recursive strategies.

## 2. Validation

- **Hold-out tail** — simple and common.
- **Rolling-origin CV** — more stable estimates (optional extension).

Do **not** shuffle rows; time order matters.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path("data/synthetic_series.csv")
META_PATH = Path("data/synthetic_series_meta.json")
df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
df = df.set_index("date").asfreq("D")
m = int(json.load(open(META_PATH, encoding="utf-8")).get("seasonal_period", 7)) if META_PATH.exists() else 7

y = df["y"].astype(float)
has_exog = {"temp", "promo"}.issubset(df.columns)
TEST_H = 120
MAX_LAG = 21
ROLL = m * 3

In [ ]:
def build_features(series: pd.Series, seasonal_period: int) -> pd.DataFrame:
    idx = series.index
    out = pd.DataFrame(index=idx)
    for ell in range(1, MAX_LAG + 1):
        out[f"lag_{ell}"] = series.shift(ell)
    s1 = series.shift(1)
    out["roll_mean"] = s1.rolling(ROLL, min_periods=seasonal_period).mean()
    out["roll_std"] = s1.rolling(ROLL, min_periods=seasonal_period).std()
    out["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
    out["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
    t = np.arange(len(series), dtype=float)
    for j in range(1, 4):
        out[f"fs_{j}"] = np.sin(2 * np.pi * j * t / seasonal_period)
        out[f"fc_{j}"] = np.cos(2 * np.pi * j * t / seasonal_period)
    return out


X = build_features(y, m)
if has_exog:
    X = pd.concat([X, df[["temp", "promo"]].astype(float)], axis=1)

tbl = X.copy()
tbl["y"] = y
tbl = tbl.dropna()

tr, te = tbl.iloc[:-TEST_H], tbl.iloc[-TEST_H:]
feature_cols = [c for c in tbl.columns if c != "y"]
X_tr, y_tr = tr[feature_cols], tr["y"]
X_te, y_te = te[feature_cols], te["y"]
print(X_tr.shape, X_te.shape)

In [ ]:
lr = LinearRegression()
lr.fit(X_tr, y_tr)
pred_lr = lr.predict(X_te)

model = CatBoostRegressor(
    depth=6,
    learning_rate=0.05,
    iterations=400,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)
model.fit(X_tr, y_tr, eval_set=Pool(X_te, y_te), early_stopping_rounds=30)
pred_cb = model.predict(X_te)


def metrics(name: str, yt: np.ndarray, yp: np.ndarray) -> None:
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mae_v = mean_absolute_error(yt, yp)
    print(f"{name:16s}  RMSE={rmse:.4f}  MAE={mae_v:.4f}")


metrics("LinearRegression", y_te.values, pred_lr)
metrics("CatBoost", y_te.values, pred_cb)

In [ ]:
imp = pd.Series(model.get_feature_importance(), index=feature_cols).sort_values(ascending=False)
ax = imp.head(15).plot.barh(figsize=(8, 4))
ax.invert_yaxis()
plt.title("CatBoost feature importance (top 15)")
plt.tight_layout()
plt.show()
imp.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_te.index, y_te.values, "k", lw=1, label="Actual")
ax.plot(y_te.index, pred_lr, lw=0.9, alpha=0.85, label="Linear")
ax.plot(y_te.index, pred_cb, lw=0.9, alpha=0.85, label="CatBoost")
ax.legend()
ax.set_title("Test period (one row per day; features use past only)")
plt.tight_layout()
plt.show()

## Interview checklist

1. **Features** — only past \(y\) and **known** future covariates.
2. **Horizon** — this setup is **direct one-step** on the test dates; multi-step needs recursion or direct \(h\)-step targets.
3. **Tuning** — use time-based CV, not random `train_test_split`.
4. **Baseline** — CatBoost should beat naive / linear if signal exists.

**Next:** Lab 6 — RNN / LSTM / GRU.